In [ ]:
import os, sys, h5py
import time

import numpy as np
import skimage.transform as transform
import matplotlib.pyplot as plt
import matplotlib.colors

import seaborn as sns
sns.set_theme()

from IPython.display import clear_output

if os.path.exists('/home/ayyerkar/.local/dragonfly/utils/py_src/'):
    sys.path.append('/home/ayyerkar/.local/dragonfly/utils/py_src/')
    print('Appended Dragonfly!')
    import detector
    import reademc
    import writeemc
else:
    print('Dragonfly not found, please install Dragonfly!')
    
det_file = 'emc/make_detector/det_agipd_9_kev.h5'

det = detector.Detector(det_fname=det_file, mask_flag=True, keep_mask_1=True)

writeEMC = False
writeEMCWaterOnly = False

<h2> Writing EMC / EMCWaterOnly files </h2>

For the AGIPD simulations, the edge pixel is technically "off the detector" for the bottom horizontal edge. This only really affects the edge resolution number as defined for PRTF/FSC. 
The solution is to crop the patterns to 90 by 90 for this data. The PRTF will never really intersect the 1/e threshold at the edge resolution for all the experimental conditions I am looking
at, so this is not really that big of an issue. And this would only affect the plot when the resolution limits are displayed. 

In [ ]:
if writeEMC:
    pat_num = '50k' # 5k or 50k or 500k
    type_ext = ['masked', 'with_water_masked']
    
    min_run, max_run = 0, 5
    for ext in type_ext:
        sys.stderr.write(f'Converting {ext} now...\n')
        for r in range(min_run, max_run):
            sys.stderr.write(f'Writing emc file for run {r+1}/{max_run}...\n')
            simulation_run = f'protein_in_water/debug_random_run_{r}_protein_in_water_{pat_num}_pats/poisson_prot_{ext}.npy'
            simulation_number = simulation_run.split(sep='/')[1]
            patterns = np.load(simulation_run)
            if patterns.dtype == 'float64':
                patterns = patterns.astype(np.int64)
            out_fname = f'sparse_frames/'+f'{simulation_number}'+'_'+f'{ext}.emc'
            wemc = writeemc.EMCWriter(out_fname, patterns.shape[1]*patterns.shape[2], hdf5=False)
            for f in range(patterns.shape[0]):
                frame = patterns[f].ravel()
                frame[frame < 0.] = 0.
                wemc.write_frame(frame)
                sys.stderr.write('\r%d/%d'%(f+1, patterns.shape[0]))
            sys.stderr.write('\n')
            wemc.finish_write()
            clear_output(wait=False)
        sys.stderr.write(f'Finished writing {max_run-min_run} file(s)...\n')

if writeEMCWaterOnly:
    pat_num = '50k' # 5k or 50k or 500k
    
    min_run, max_run = 0,5
    for r in range(min_run, max_run):
        sys.stderr.write(f'Writing emc background file for run {r+1}/{max_run}...\n')
        simulation_run = f'water_only/debug_random_run_{r}_water_{pat_num}_pats/poisson_water_only_masked.npy'
        simulation_number = simulation_run.split(sep='/')[1]
        patterns = np.load(simulation_run)
        if patterns.dtype == 'float64':
            patterns = patterns.astype(np.int64)
        out_fname = f'sparse_frames_water_only/'+f'{simulation_number}'+'_'+'poisson_water_only_masked.emc'
        wemc = writeemc.EMCWriter(out_fname, patterns.shape[1]*patterns.shape[2], hdf5=False)
        for f in range(patterns.shape[0]): 
            frame = patterns[f].ravel()
            frame[frame < 0.] = 0.
            wemc.write_frame(frame)
            sys.stderr.write('\r%d/%d'%(f+1, patterns.shape[0]))
        sys.stderr.write('\n')
        wemc.finish_write()
        clear_output(wait=False)
    sys.stderr.write(f'Finished writing {max_run-min_run} file(s)...\n')

<h2> Inspecting written EMC files with no water diffraction </h2> 

In [ ]:
run_nr = 0
npats = '50k'
ext = 'masked'

emc_1 = reademc.EMCReader([f'sparse_frames/debug_random_run_{run_nr}_protein_in_water_{npats}_pats_{ext}.emc'], geom_list=[det])
emc_2 = reademc.EMCReader([f'sparse_frames/debug_random_run_{run_nr}_protein_in_water_{npats}_pats_{ext}.emc'], geom_list=[det]) 
emc_3 = reademc.EMCReader([f'sparse_frames/debug_random_run_{run_nr}_protein_in_water_{npats}_pats_{ext}.emc'], geom_list=[det]) 

rng = np.random.default_rng()
r_pat1 = rng.integers(0,10000)
r_pat2 = rng.integers(0,10000)
r_pat3 = rng.integers(0,10000)

emc_frame1 = emc_1.get_frame(r_pat1,sym=False)
emc_frame2 = emc_2.get_frame(r_pat2,sym=False)
emc_frame3 = emc_3.get_frame(r_pat3,sym=False)

fig_handle = plt.figure(constrained_layout = True, dpi = 200)
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
min_v = 0
max_v = 5
cm = plt.get_cmap('viridis', max_v+1)

ax_0 = fig_handle.add_subplot(spec_handle[0,0])
im_0 = plt.imshow(emc_frame1,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
ax_0.set_title(f'run {run_nr} - {r_pat1} ({emc_frame1.sum()} phs)',weight='bold', size=8)
ax_0.set_xticks([])
ax_0.set_yticks([])
c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.05,shrink=0.43,orientation='vertical')
c_bar_0.set_ticks(np.arange(min_v, max_v+1))

ax_1 = fig_handle.add_subplot(spec_handle[0,1])
im_1 = plt.imshow(emc_frame2,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
ax_1.set_title(f'run {run_nr} - {r_pat2} ({emc_frame2.sum()} phs)',weight='bold', size=8)
ax_1.set_xticks([])
ax_1.set_yticks([])

ax_2 = fig_handle.add_subplot(spec_handle[0,2])
im_2 = plt.imshow(emc_frame3,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
ax_2.set_title(f'run {run_nr} - {r_pat3} ({emc_frame3.sum()} phs)',weight='bold', size=8)
ax_2.set_xticks([])
ax_2.set_yticks([]);

<h2> Inspecting written EMC files with water diffraction </h2> 

In [ ]:
run_nr = 1
npats = '50k'
ext = 'with_water_masked'

emc_1 = reademc.EMCReader([f'sparse_frames/debug_random_run_{run_nr}_protein_in_water_{npats}_pats_{ext}.emc'], geom_list=[det])
emc_2 = reademc.EMCReader([f'sparse_frames/debug_random_run_{run_nr}_protein_in_water_{npats}_pats_{ext}.emc'], geom_list=[det]) 
emc_3 = reademc.EMCReader([f'sparse_frames/debug_random_run_{run_nr}_protein_in_water_{npats}_pats_{ext}.emc'], geom_list=[det]) 

rng = np.random.default_rng()
r_pat1 = rng.integers(0,10000)
r_pat2 = rng.integers(0,10000)
r_pat3 = rng.integers(0,10000)

emc_frame1 = emc_1.get_frame(r_pat1,sym=False)
emc_frame2 = emc_2.get_frame(r_pat2,sym=False)
emc_frame3 = emc_3.get_frame(r_pat3,sym=False)

fig_handle = plt.figure(constrained_layout = True, dpi = 200)
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
min_v = 0
max_v = 5
cm = plt.get_cmap('viridis', max_v+1)

ax_0 = fig_handle.add_subplot(spec_handle[0,0])
im_0 = plt.imshow(emc_frame1,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
#ax_0.set_title(f'run {run_nr} - {r_pat1} ({emc_frame1.sum()} phs)',weight='bold', size=8)
ax_0.set_title(f'({emc_frame1.sum()} phs)',weight='bold', size=8)
ax_0.set_xticks([])
ax_0.set_yticks([])
c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.05,shrink=0.43,orientation='vertical')
c_bar_0.set_ticks(np.arange(min_v, max_v+1))

ax_1 = fig_handle.add_subplot(spec_handle[0,1])
im_1 = plt.imshow(emc_frame2,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
#ax_1.set_title(f'run {run_nr} - {r_pat2} ({emc_frame2.sum()} phs)',weight='bold', size=8)
ax_1.set_title(f'({emc_frame2.sum()} phs)',weight='bold', size=8)
ax_1.set_xticks([])
ax_1.set_yticks([])

ax_2 = fig_handle.add_subplot(spec_handle[0,2])
im_2 = plt.imshow(emc_frame3,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
#ax_2.set_title(f'run {run_nr} - {r_pat3} ({emc_frame3.sum()} phs)',weight='bold', size=8)
ax_2.set_title(f'({emc_frame3.sum()} phs)',weight='bold', size=8)
ax_2.set_xticks([])
ax_2.set_yticks([]);

<h2> Inspecting written EMC files with only water diffraction </h2> 

In [ ]:
run_nr = 0
npats = '50k'
ext = 'poisson_water_only_masked'

emc_1 = reademc.EMCReader([f'sparse_frames_water_only/debug_random_run_{run_nr}_water_{npats}_pats_{ext}.emc'], geom_list=[det])
emc_2 = reademc.EMCReader([f'sparse_frames_water_only/debug_random_run_{run_nr}_water_{npats}_pats_{ext}.emc'], geom_list=[det]) 
emc_3 = reademc.EMCReader([f'sparse_frames_water_only/debug_random_run_{run_nr}_water_{npats}_pats_{ext}.emc'], geom_list=[det]) 
_
rng = np.random.default_rng()
r_pat1 = rng.integers(0,10000)
r_pat2 = rng.integers(0,10000)
r_pat3 = rng.integers(0,10000)

emc_frame1 = emc_1.get_frame(r_pat1,sym=False)
emc_frame2 = emc_2.get_frame(r_pat2,sym=False)
emc_frame3 = emc_3.get_frame(r_pat3,sym=False)

fig_handle = plt.figure(constrained_layout = True, dpi = 200)
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
min_v = 0
max_v = 2
cm = plt.get_cmap('viridis', max_v+1)

ax_0 = fig_handle.add_subplot(spec_handle[0,0])
im_0 = plt.imshow(emc_frame1,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
#ax_0.set_title(f'run {run_nr} - {r_pat1} ({emc_frame1.sum()} phs)',weight='bold', size=8)
ax_0.set_title(f'({emc_frame1.sum()} phs)',weight='bold', size=8)
ax_0.set_xticks([])
ax_0.set_yticks([])
c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.05,shrink=0.43,orientation='vertical')
c_bar_0.set_ticks(np.arange(min_v, max_v+1))

ax_1 = fig_handle.add_subplot(spec_handle[0,1])
im_1 = plt.imshow(emc_frame2,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
#ax_1.set_title(f'run {run_nr} - {r_pat2} ({emc_frame2.sum()} phs)',weight='bold', size=8)
ax_1.set_title(f'({emc_frame2.sum()} phs)',weight='bold', size=8)
ax_1.set_xticks([])
ax_1.set_yticks([])

ax_2 = fig_handle.add_subplot(spec_handle[0,2])
im_2 = plt.imshow(emc_frame3,vmin=min_v-0.5,vmax=max_v+0.5,cmap=cm,interpolation=None)
#ax_2.set_title(f'run {run_nr} - {r_pat3} ({emc_frame3.sum()} phs)',weight='bold', size=8)
ax_2.set_title(f'({emc_frame3.sum()} phs)',weight='bold', size=8)
ax_2.set_xticks([])
ax_2.set_yticks([]);

Below the powder sum of the water background is also shown for reference. First cell is for a run with 5k patterns, and second cell is for a run with 50k patterns. 

In [ ]:
emc_5k  = reademc.EMCReader([f'sparse_frames_water_only/random_run_{run_nr}_water_5k_pats_{ext}.emc'], geom_list=[det])
emc_50k = reademc.EMCReader([f'sparse_frames_water_only/random_run_{run_nr}_water_50k_pats_{ext}.emc'], geom_list=[det])

emc_5k_powder = emc_5k.get_powder()
emc_50k_powder = emc_50k.get_powder()

fig_handle = plt.figure(constrained_layout = True, dpi = 200)
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 2)
cm = 'viridis'

ax_0 = fig_handle.add_subplot(spec_handle[0,0])
maxv = 30
im_0 = plt.imshow(emc_5k_powder,vmin=0,vmax=maxv,cmap=cm,interpolation=None)
ax_0.set_title(f'run {run_nr} - ({emc_5k_powder.sum()} phs)',weight='bold', size=8)
ax_0.set_xticks([])
ax_0.set_yticks([])
c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.05,shrink=0.53,orientation='vertical')
c_bar_0.set_ticks([0, maxv/2, maxv])

ax_1 = fig_handle.add_subplot(spec_handle[0,1])
maxv_2 = maxv * 10
im_1 = plt.imshow(emc_50k_powder,vmin=0,vmax=maxv_2,cmap=cm,interpolation=None)
ax_1.set_title(f'run {run_nr} - ({emc_50k_powder.sum()} phs)',weight='bold', size=8)
ax_1.set_xticks([])
ax_1.set_yticks([])
c_bar_1 = plt.colorbar(im_1, ax=ax_1,fraction=0.05,shrink=0.53,orientation='vertical')
c_bar_1.set_ticks([0, maxv_2/2, maxv_2]);

<h2> Checking beta parameter schedule and rotational sampling </h2> 
Note that Dragonfly determines the beta schedule by modulo dividing the number of iterations by the jump schedule.
This means that if the remainder is not equal to 0, there will be a few iterations with a higher beta than as set
by the schedule. However, for beta equal to 1 - so no annealing. 

In [ ]:
beta = 0.001

beta_jump, beta_schedule = 1.4141, 15
max_iter = 250
b_list = [beta]
max_iter = max_iter//beta_schedule - 1

for i in range(max_iter):
    beta*=beta_jump
    b_list.append(beta)
    
plotBeta = True
if plotBeta:
    plt.figure(dpi=100)
    plt.plot(b_list,'o--')
    plt.xticks(ticks=np.arange(0,max_iter+1,3))
    plt.yticks(ticks=[b_list[0],b_list[-1]])
    plt.xlabel('jump iteration')
    plt.ylabel('beta');
    print(f'Beta schedule: {b_list}')
else:
    num_div = 3
    num_rot = 50*(num_div)**3+10*num_div
    print(f'Rotational samples: {num_rot}')

In [ ]:
n = np.arange(1, 14, 1)

num_rot = 50*(n**3)+10*n
print(num_rot, end='\n\n')
delta_rot = 0.944 / n
print(delta_rot)

fig_handle = plt.figure(constrained_layout = True, dpi = 110)
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 2)

ax_0 = fig_handle.add_subplot(spec_handle[0])
plt.plot(n, num_rot, 'b')
plt.xlabel('n', weight='bold')
plt.ylabel('num_rot', weight='bold')
plt.xticks(n[::2])
plt.ylim([-2e3, None])

ax_1 = fig_handle.add_subplot(spec_handle[1])
plt.plot(n, delta_rot, 'r')
plt.xlabel('n', weight='bold')
plt.ylabel('delta_rot', weight='bold')
plt.xticks(n[::2])
plt.ylim([0, 1]);